## Live Parser → get_processed_data(source="live") — проверка каждую секунду

Запускается LiveCollector (parser), данные собираются через **get_processed_data(source="live", live_collector=parser, grid_resolution_ms=5000)** с сеткой 5 секунд. Каждую секунду выводятся текущее время и последнее значение **exchange_ts** в датафрейме.

In [ ]:
import asyncio
from datetime import datetime
from src.parser.live_collector import LiveCollector
from src.mlcore.dataloader import get_processed_data

SYMBOL = "BTCUSDC"
GRID_RESOLUTION_MS = 5000  # 5 секунд
RETENTION_SECONDS = 300

parser = LiveCollector(symbol=SYMBOL, retention_seconds=RETENTION_SECONDS)
parser_task = asyncio.create_task(parser.start())
print("🟢 LiveCollector started. Waiting for data...")

In [ ]:
# Каждую секунду: get_processed_data(source="live", ...) и вывод текущего времени + последний exchange_ts
try:
    for step in range(60):  # 60 секунд; прервать ядро для остановки
        df = get_processed_data(
            source="live",
            live_collector=parser,
            grid_resolution_ms=GRID_RESOLUTION_MS,
        )
        now = datetime.utcnow()
        last_ts = df["exchange_ts"].iloc[-1] if not df.empty and "exchange_ts" in df.columns else None
        if last_ts is not None and hasattr(last_ts, "strftime"):
            last_ts_str = last_ts.strftime("%H:%M:%S")
        elif last_ts is not None:
            last_ts_str = str(last_ts)
        else:
            last_ts_str = "(no data)"
        print(f"now UTC = {now.strftime('%H:%M:%S')}  |  last exchange_ts = {last_ts_str}  |  rows = {len(df)}")
        await asyncio.sleep(1)
except asyncio.CancelledError:
    pass
print("Stopped.")

In [ ]:
# Остановить парсер (выполнить после остановки цикла выше)
parser_task.cancel()
try:
    await parser_task
except asyncio.CancelledError:
    pass
await parser.stop()
print("🛑 Parser stopped.")